## ファイルの前処理

In [1]:
import os, requests
from Bio import PDB
from pdbfixer import PDBFixer
from openmm.app import PDBFile

In [2]:
base_dir = "/home/shaeo/cadd_training/20250923_OepnMM"

In [3]:
# ファイルのダウンロード（4ZJ8: OX1）
pdb_id = "4ZJC"
pdb_id = pdb_id.lower()
url = f"https://biomembhub.org/shared/opm-assets/pdb/{pdb_id}.pdb"
os.makedirs(base_dir + "/data", exist_ok=True)
out_file = base_dir + f"/data/{pdb_id}_opm.pdb"

response = requests.get(url)
if response.status_code == 200:
    with open(out_file, "wb") as f:
        f.write(response.content)
else:
    print(f"Failed to fetch {pdb_id} from OPM (status {response.status_code})")

In [4]:
# リガンドとレセプターの分割
protein_file = base_dir + f"/data/receptor_{pdb_id}.pdb"
ligand_file = base_dir + f"/data/ligand_{pdb_id}.pdb"
ligand_resname = "4OT"

parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure("complex", out_file)
io = PDB.PDBIO()

# レセプターを抽出
class ProteinSelect(PDB.Select):
    def accept_residue(self, residue):
        if not PDB.is_aa(residue, standard=True): # アミノ酸かどうか
            return False
        res_id = residue.id[1]
        if 1001 <= res_id <= 1196: # 残基番号を取得 (id[1] が residue number)
            return False  # この範囲は除外
        return True
io.set_structure(structure)
io.save(protein_file, ProteinSelect())

# リガンドを抽出
class LigandSelect(PDB.Select):
    def accept_residue(self, residue):
        return residue.get_resname().strip() == ligand_resname
io.set_structure(structure)
io.save(ligand_file, LigandSelect())

In [5]:
# リガンドのファイル形式を変換
ligand_sdf = base_dir + f"/data/ligand_{pdb_id}.sdf"
!obabel $ligand_file -O $ligand_sdf -p 7.4

1 molecule converted


In [6]:
# PDBfixer
protein_file_fixed = base_dir + f"/data/receptor_{pdb_id}-fixed.pdb"

fixer = PDBFixer(filename=protein_file)
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.4)

with open(protein_file_fixed, "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)

## 複合体の準備

In [7]:
import numpy as np
from rdkit import Chem
import mdtraj as md
import openmm as mm
import openmm.app as app
from openmm import unit
from openff.toolkit.topology import Molecule
from openmmforcefields.generators import GAFFTemplateGenerator

In [8]:
# リガンドの前処理
ligname = "LIG"
supplier = Chem.SDMolSupplier(ligand_sdf, removeHs=False)
rdkit_mol = supplier[0]

# OpenFFのオブジェクトに変換
off_mol = Molecule.from_rdkit(rdkit_mol)    
off_mol.name = ligname  # リガンドの名前を指定

# 原子番号の指定
element_counter_dict = {}
for off_atom, rdkit_atom in zip(off_mol.atoms, rdkit_mol.GetAtoms()):
    element = rdkit_atom.GetSymbol()
    if element in element_counter_dict.keys():
        element_counter_dict[element] += 1
    else:
        element_counter_dict[element] = 1
    off_atom.name = element + str(element_counter_dict[element])

# OpenMMのオブジェクトに変換
off_mol_topology = off_mol.to_topology()
mol_topology = off_mol_topology.to_openmm()
mol_positions = off_mol.conformers[0]
mol_positions = mol_positions.to("nanometers")  # 単位を変換：Å → nm
omm_mol = app.Modeller(mol_topology, mol_positions)

In [9]:
# rガンド-タンパク質複合体の形成
md_protein_topology = md.Topology.from_openmm(fixer.topology)  # PDBfixerのオブジェクトを利用
md_ligand_topology = md.Topology.from_openmm(omm_mol.topology)
md_complex_topology = md_protein_topology.join(md_ligand_topology)
complex_topology = md_complex_topology.to_openmm()  # OpenMMのオブジェクトに変換

# 座標の更新
total_atoms = len(fixer.positions) + len(omm_mol.positions)
complex_positions = unit.Quantity(np.zeros([total_atoms, 3]), unit=unit.nanometers) # OpenMMで利用される配列
complex_positions[: len(fixer.positions)] = fixer.positions # タンパク質の座標を追加
complex_positions[len(fixer.positions) :] = omm_mol.positions   # リガンドの座標を追加

/home/shaeo/miniconda3/envs/openmm/lib/python3.10/site-packages/openmm/unit/quantity.py:752: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  self._value[key] = value / self.unit


In [10]:
# 力場の設定
protein_ff="amber14-all.xml"    # タンパク質の力場（膜脂質の力場を含む）
solvent_ff="amber14/tip3pfb.xml"    # 水の力場
forcefield = app.ForceField(protein_ff, solvent_ff) 
gaff = GAFFTemplateGenerator(
    molecules=Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True) # リガンドの力場/電荷を設定
)
forcefield.registerTemplateGenerator(gaff.generator)

In [11]:
# 複合体に水/イオン/膜脂質を追加
modeller = app.Modeller(complex_topology, complex_positions)
modeller.addMembrane(
    forcefield,
    lipidType="POPC",
    membraneCenterZ=0.0*unit.nanometer,  # 膜中心の Z 位置
    minimumPadding=1.0*unit.nanometer,    # タンパク質等との余裕 (10Å)
    positiveIon="Na+",
    negativeIon="Cl-",              
    ionicStrength=0.15*unit.molar
)

In [12]:
# PDBファイルとして保存
complex_file = base_dir + f"/data/complex_{pdb_id}.pdb"
with open(complex_file, "w") as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)

## MDシミュレーション

In [13]:
result_dir = base_dir + "/result"
os.makedirs(result_dir, exist_ok=True)

In [14]:
# simulationオブジェクトを作成

# システム
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints='HBonds'
)

# バロスタット
barostat = mm.MonteCarloMembraneBarostat(
    1*unit.bar,                      # 圧力
    0*unit.bar*unit.nanometer,       # 表面張力（通常は0）
    300*unit.kelvin,                 # 温度
    mm.MonteCarloMembraneBarostat.XYIsotropic,
    mm.MonteCarloMembraneBarostat.ZFree,
    25                                # 頻度
)
system.addForce(barostat)

# インテグレーター
integrator = mm.LangevinIntegrator(
    10*unit.kelvin,
    1/unit.picosecond,
    0.002*unit.picoseconds
)

# プラットフォーム
platform = mm.Platform.getPlatformByName('CUDA')  # または 'OpenCL', 'CPU'
properties = {'Precision': 'mixed'} # GPUの計算精度

# トポロジーと座標の取得
topology = modeller.getTopology()
position = modeller.getPositions()

simulation = app.Simulation(topology, system, integrator, platform, properties)
simulation.context.setPositions(position)

### 1.エネルギー最適化

In [15]:
state_path_em = result_dir + "/em_state.xml"
structure_path_em = result_dir + "/em_structure.pdb"

In [16]:
# 実行
simulation.minimizeEnergy()

In [17]:
# 保存
state_em = simulation.context.getState(
    getPositions=True, 
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_em, "w") as f:
    f.write(mm.XmlSerializer.serialize(state_em))
position_em = simulation.context.getState(getPositions=True).getPositions()
with open(structure_path_em, "w") as f:
    PDBFile.writeFile(simulation.topology, position_em, f)

### 2.等温緩和

In [18]:
state_path_qd = result_dir + "/qd_state.xml"
dcd_path_qd = result_dir + "/qd_traj.dcd"
log_path_qd = result_dir + "/qd_log.log"

In [19]:
# 設定
simulation.currentStep = 0
simulation.reporters = []
simulation.reporters.append(app.DCDReporter(dcd_path_qd, 2))
simulation.reporters.append(
    app.StateDataReporter(
        log_path_qd, 2, totalSteps=10, step=True, speed=True, progress=True, potentialEnergy=True, temperature=True
    )
)
start_velocity = 10*unit.kelvin
simulation.context.setVelocitiesToTemperature(start_velocity)

In [20]:
# 実行
simulation.step(10)

In [21]:
# 保存
state_qd = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_qd, "w") as f:
    f.write(mm.XmlSerializer.serialize(state_qd))

### 3.昇温

In [22]:
state_path_ht = result_dir + "/ht_state.xml"
dcd_path_ht = result_dir + "/ht_traj.dcd"
log_path_ht = result_dir + "/ht_log.log"

In [23]:
# 設定
n_steps = 250000  # 2fs * 250,000 = 500,000fs ← 500ps ← 0.5ns
simulation.currentStep = 0
simulation.reporters = []
simulation.reporters.append(app.DCDReporter(dcd_path_ht, 5000))
simulation.reporters.append(
    app.StateDataReporter(
        log_path_ht, 5000, totalSteps=n_steps, step=True, speed=True, progress=True, potentialEnergy=True, temperature=True
    )
)

In [24]:
# 拘束条件: 脂質、タンパク主鎖、リガンド
restraint = mm.CustomExternalForce("k*periodicdistance(x, y, z, x0, y0, z0)^2")
res_idx = system.addForce(restraint)
restraint.addGlobalParameter("k", 1000.0*unit.kilojoules_per_mole/unit.nanometer)
restraint.addPerParticleParameter("x0")
restraint.addPerParticleParameter("y0")
restraint.addPerParticleParameter("z0")
lipid_atoms = [atom for atom in topology.atoms() if atom.residue.name=="POP"]
for atom in lipid_atoms:
    restraint.addParticle(atom.index, position[atom.index])
protein_atoms = [atom for atom in topology.atoms() if atom.name in ["CA", "N", "C", "O"]]
for i in protein_atoms:
   restraint.addParticle(atom.index, position[atom.index])
ligand_atoms = [atom for atom in topology.atoms() if atom.residue.name=="LIG"]
for i in protein_atoms:
   restraint.addParticle(atom.index, position[atom.index])

In [25]:
# 実行
list_temp = [100, 150, 200, 250, 300]
list_step = [50000, 50000, 50000, 50000, 50000]
for temp, step in zip(list_temp, list_step):
    integrator.setTemperature(temp*unit.kelvin)
    barostat.setDefaultTemperature(temp*unit.kelvin)
    simulation.step(step)

In [26]:
# 保存
state_ht = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_ht, "w") as f:
    f.write(mm.XmlSerializer.serialize(state_ht))

### 4.平衡化

In [27]:
state_path_eq = result_dir + "/eq_state.xml"
dcd_path_eq = result_dir + "/eq_traj.dcd"
log_path_eq = result_dir + "/eq_log.log"

In [28]:
# 設定
n_steps = 250000  # 2fs * 250,000 = 500,000fs ← 500ps ← 0.5ns
simulation.currentStep = 0
simulation.reporters = []
simulation.reporters.append(app.DCDReporter(dcd_path_eq, 5000))
simulation.reporters.append(
    app.StateDataReporter(
        log_path_eq, 5000, totalSteps=n_steps, step=True, speed=True, progress=True, potentialEnergy=True, temperature=True
    )
)

In [29]:
# 実行
list_restrain = [1000.0, 300.0, 100.0, 30.0, 0.0]
list_step = [5000, 40000, 40000, 40000, 125000]
for k, step in zip(list_restrain, list_step):
    restraint.setGlobalParameterDefaultValue(0, k*unit.kilojoules_per_mole/unit.nanometer)  
    simulation.step(step)

In [30]:
# 保存
state_eq = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_eq, "w") as f:
    f.write(mm.XmlSerializer.serialize(state_eq))

### 5.生産

In [31]:
state_path_md = result_dir + "/md_state.xml"
dcd_path_md = result_dir + "/md_traj.dcd"
log_path_md = result_dir + "/md_log.log"
checkpoint_path = result_dir + "/md_checkpoint.chk"

In [32]:
# 設定
n_steps = 500000  # 2fs * 500,000 = 1,000,000fs ← 1000ps ← 1.0ns
simulation.currentStep = 0
simulation.reporters = []
simulation.reporters.append(app.DCDReporter(dcd_path_md, 5000))
simulation.reporters.append(
    app.StateDataReporter(
        log_path_md, 5000, totalSteps=n_steps, step=True, speed=True, progress=True, potentialEnergy=True, temperature=True
    )
)

In [33]:
# 実行
simulation.step(n_steps)

In [34]:
# 保存
state_md = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_md, "w") as f:
    f.write(mm.XmlSerializer.serialize(state_md))
with open(checkpoint_path, "wb") as f:
    f.write(simulation.context.createCheckpoint())

### 6.トラジェクトリの後処理

In [35]:
fixed_path_ht = base_dir + "/result/ht_traj_pc.dcd"
fixed_path_eq = base_dir + "/result/eq_traj_pc.dcd"
fixed_path_md = base_dir + "/result/md_traj_pc.dcd"

In [36]:
dict_path_dcd = {
    dcd_path_ht:fixed_path_ht,
    dcd_path_eq:fixed_path_eq,
    dcd_path_md:fixed_path_md,
}

In [37]:
# PBC補正
for dcd_path, fixed_dcd_path in dict_path_dcd.items():
    traj = md.load(dcd_path, top=structure_path_em)
    traj_whole = traj.make_molecules_whole()    # 分子を “ちぎれない” 形に直す（make whole）
    prot = traj_whole.topology.select("protein")    # タンパクを基準にセンタリング＆ボックス内にラップ
    traj_centered = traj_whole.center_coordinates()
    traj_wrapped = traj_centered.image_molecules()  # 近接イメージに押し戻す
    traj_fit = traj_wrapped.superpose(traj_wrapped, frame=0, atom_indices=prot)     # 剛体合わせ（平行移動・回転除去）してRMSD等が滑らかに
    traj_fit.save_dcd(fixed_dcd_path)

## MM/GBSA、MM/PBSA

In [96]:
import parmed as pmd
from parmed.tools.actions import changeRadii

In [79]:
complex_prmtop_path = base_dir + "/result/complex.prmtop"
complex_inpcrd_path = base_dir + "/result/complex.inpcrd"
receptor_prmtop_path = base_dir + "/result/receptor.prmtop"
receptor_inpcrd_path = base_dir + "/result/receptor.inpcrd"
ligand_prmtop_path = base_dir + "/result/ligand.prmtop"
ligand_inpcrd_path = base_dir + "/result/ligand.inpcrd"
dcd_mm_path = base_dir + "/result/md_traj_for_mm.dcd"

In [108]:
# ParmEd 変換
modeller.deleteWater()  # 水分子を削除
to_delete = [res for res in modeller.topology.residues()
             if res.name in ["NA", "CL", "POP"]]
modeller.delete(to_delete)  # 特定の残基を削除
_topology, _positiion = modeller.topology, modeller.positions
_forcefield = app.ForceField(protein_ff, solvent_ff) 
_gaff = GAFFTemplateGenerator(
    molecules=Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True) # リガンドの力場/電荷を設定
)
_forcefield.registerTemplateGenerator(_gaff.generator)
_system = _forcefield.createSystem(
    _topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints='HBonds'
)
parm = pmd.openmm.load_topology(_topology, _system, _positiion)
parm.box = None
changeRadii(parm, 'mbondi2').execute()

# 複合体を保存
parm.save(complex_prmtop_path, overwrite=True)
parm.save(complex_inpcrd_path, overwrite=True)

# 受容体のみを選択
parm_receptor = parm['(:1-307)']    # result/em_structure.pdb
parm_receptor.save(receptor_prmtop_path, overwrite=True)
parm_receptor.save(receptor_inpcrd_path, overwrite=True)

# リガンドのみを選択（例: 残基名 LIG）
parm_ligand = parm['(:UNK)']
parm_ligand.save(ligand_prmtop_path, overwrite=True)
parm_ligand.save(ligand_inpcrd_path, overwrite=True)

In [80]:
traj = md.load_dcd(fixed_path_md, top=structure_path_em)
# タンパク + リガンドのみ残す
prot_lig = traj.atom_slice(traj.topology.select("protein or resname UNK"))
prot_lig.save_dcd(dcd_mm_path)

### MM/GBSA

In [109]:
mmgbsa_in_path = base_dir + "/result/mmgbsa.in"
mmgbsa_result_path = base_dir + "/result/mmgbsa.dat"

In [110]:
mmgbsa_in = '''
&general
   startframe=1, 
   endframe=1000000,
   interval=1,
   keep_files=0, 
   verbose=1,
/
&gb
   igb=5, 
   saltcon=0.150,
/
'''
with open(mmgbsa_in_path, "w") as f:
    f.write(mmgbsa_in)

In [112]:
!MMPBSA.py -O -i $mmgbsa_in_path -o $mmgbsa_result_path \
  -cp $complex_prmtop_path \
  -rp $receptor_prmtop_path \
  -lp $ligand_prmtop_path \
  -y $dcd_mm_path

Loading and checking parameter files for compatibility...
cpptraj found! Using /home/shaeo/miniconda3/envs/openmm/bin/cpptraj
mmpbsa_py_energy found! Using /home/shaeo/miniconda3/envs/openmm/bin/mmpbsa_py_energy
Preparing trajectories for simulation...
100 frames were processed by cpptraj for use in calculation.

Running calculations on normal system...

Beginning GB calculations with /home/shaeo/miniconda3/envs/openmm/bin/mmpbsa_py_energy
  calculating complex contribution...
  calculating receptor contribution...
  calculating ligand contribution...

Timing:
Total setup time:                           0.007 min.
Creating trajectories with cpptraj:         0.012 min.
Total calculation time:                     2.823 min.

Total GB calculation time:                  2.823 min.

Statistics calculation & output writing:    0.000 min.
Total time taken:                           2.841 min.


MMPBSA.py Finished! Thank you for using. Please cite us if you publish this work with this paper:
 

### MM/PBSA

In [114]:
mmpbsa_in_path = base_dir + "/result/mmpbsa.in"
mmpbsa_result_path = base_dir + "/result/mmpbsa.dat"

In [115]:
mmpbsa_in = '''
&general
   startframe=1, 
   endframe=10000, 
   interval=5,
   keep_files=0, 
   verbose=1,
/
&pb
   istrng=0.150,
   indi=20.0, 
   exdi=80.0,
   inp=2, 
   radiopt=0,
   fillratio=4.0, 
   scale=2.0,
   ! --- implicit membrane ---
   memopt=1,
   emem=7.0,
   mthick=15.8,
   mctrdz=0.0,
   poretype=1,
/
'''
with open(mmpbsa_in_path, "w") as f:
    f.write(mmpbsa_in)

In [116]:
!MMPBSA.py -O -i $mmpbsa_in_path -o $mmpbsa_result_path \
  -cp $complex_prmtop_path \
  -rp $receptor_prmtop_path \
  -lp $ligand_prmtop_path \
  -y $dcd_mm_path

Loading and checking parameter files for compatibility...
cpptraj found! Using /home/shaeo/miniconda3/envs/openmm/bin/cpptraj
mmpbsa_py_energy found! Using /home/shaeo/miniconda3/envs/openmm/bin/mmpbsa_py_energy
Preparing trajectories for simulation...
20 frames were processed by cpptraj for use in calculation.
Will use the membrane thickness and location defined by user manually
membrane thickness: 15.8  membrane location: 0.0

Running calculations on normal system...

Beginning PB calculations with /home/shaeo/miniconda3/envs/openmm/bin/mmpbsa_py_energy
  calculating complex contribution...
  calculating receptor contribution...
  calculating ligand contribution...

Timing:
Total setup time:                           0.007 min.
Creating trajectories with cpptraj:         0.004 min.
Total calculation time:                     7.446 min.

Total PB calculation time:                  7.446 min.

Statistics calculation & output writing:    0.000 min.
Total time taken:                     